# Build and inspect an energy network

Build the energy network products (optionally, from this notebook or from the terminal) and inspect **one** of them: summary, interactive map, node and edge tables, validation report and the PyPSA network.

The build turns the inputs reviewed in `../00-data-review/00_data_review.ipynb` into three products, each a PyPSA network with GeoParquet map layers:

- `base-mauritius` — the provided CEB transmission network: the substations snapped onto the CEB routes, with the generation sites attached.
- `inferred-osm-<region>` — OpenStreetMap substations, plants and generators joined by the OSM roads that lie near VIIRS night-light targets (the roads stand in for the distribution network).
- `inferred-provided-<region>` — the provided substations and generators plus the CEB backbone, joined by the same road network.

The region is `energy.region` in `config/energy/energy.yaml` (default `mauritius-rodrigues`). Voltages and capacities of the inferred lines are placeholders: `v_nom_kv` is deliberately empty in the published layers and `model_v_nom_kv` holds the value the model uses.

## How the build works, and how to run it

This notebook is set up to **review** built outputs; it does not build anything by default. The products come from one Snakemake target, `build_energy_networks`, defined in `workflow/0-preprocess/energy.smk`. It asks for the three `<product>.nc` files and Snakemake works out the chain of rules:

1. `prepare_energy_assets` cleans the provided CEB shapefiles and demand workbook into `data/processed/energy/provided/` (script: `energy_prepare_assets.py`).
2. `build_base_energy_network` runs `--source base` and writes `base-mauritius`.
3. `fetch_energy_osm` downloads the OpenStreetMap roads, power features and area of interest to `data/incoming/Infrastructure/Energy/OpenStreetMap/<region>/`. Snakemake runs it only when one of those files is missing, so a copy from the project's shared drive is used as it is.
4. `fetch_energy_nightlights` (likewise only when a tile is missing) → `build_energy_nightlight_composite` → `build_energy_nightlight_targets` turn the monthly VIIRS tiles into a radiance composite and then into night-light target points (`targets.geoparquet`).
5. `build_inferred_energy_network` runs once per variant: `--source inferred-osm` writes `inferred-osm-<region>` and `--source inferred-provided` writes `inferred-provided-<region>`. Both read the cached roads and the targets; the osm variant also reads the cached power features, the provided variant the cleaned CEB tables instead.

Every build rule shells out to `workflow/0-preprocess/energy_build_network.py`, a thin `click` wrapper around `energy.build.build_network(source, ...)`; the parameters come from `config/energy/energy.yaml`. Each product writes `data/processed/energy/networks/<name>/`: `<name>.nc` (PyPSA) and `<name>_metadata.json`, plus a `geoparquet/` folder with the node and edge layers and a spatial manifest (what `h.load_layers` reads below); and `data/out/energy/<name>/` with `generators.csv`, `lines.csv` and `validation.json`.

**From the terminal** (repo root, `mu-star` environment active):

```shell
snakemake -c1 build_energy_networks      # all three products
snakemake -n build_energy_networks       # dry run: list what would (re)run and why
snakemake -c1 -F build_energy_networks   # force a full rebuild
# one product: name the file it writes
snakemake -c1 data/processed/energy/networks/base-mauritius/base-mauritius.nc
snakemake -c1 data/processed/energy/networks/inferred-osm-mauritius-rodrigues/inferred-osm-mauritius-rodrigues.nc
snakemake -c1 data/processed/energy/networks/inferred-provided-mauritius-rodrigues/inferred-provided-mauritius-rodrigues.nc
```

Only `build_energy_networks` can be run by name; every other rule has a `{data}` wildcard and is asked for by the path of a file it writes (`data/` being the `data_root` in `config/config.yaml`).

**From this notebook**: set `RUN_BUILD = True` in the next cell. It runs the same Snakemake target from the repo root with this kernel's Python, so `data_root` in `config/config.yaml` resolves exactly as on the command line. Re-run the cells below afterwards to review the fresh outputs.

In [ ]:
# Optional: run the build from here. Off by default because this notebook is
# for reviewing outputs; set RUN_BUILD = True to run the same Snakemake target
# the terminal command uses (from the repo root, with this kernel's Python).
import pathlib
import subprocess
import sys

RUN_BUILD = False
# All three products. For one product name the file it writes instead, e.g.
# "data/processed/energy/networks/base-mauritius/base-mauritius.nc" (only
# build_energy_networks runs by name).
TARGET = "build_energy_networks"
EXTRA_ARGS: list[str] = []  # e.g. ["-n"] for a dry run, ["-F"] to force a full rebuild

if RUN_BUILD:
    repo_root = next(p for p in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents) if (p / "pyproject.toml").exists())
    subprocess.run([sys.executable, "-m", "snakemake", "-c1", TARGET, *EXTRA_ARGS], cwd=repo_root, check=True)

## Pick a product

The next cell lists the products that are built (a product counts once its `geoparquet/` layers and manifest exist) and selects one by index. Change `PRODUCT_INDEX` and re-run from here to look at another.

In [ ]:
import pathlib
import sys

for _candidate in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_candidate / "_helpers.py").exists():
        sys.path.insert(0, str(_candidate))
        break
# _helpers is a dev-only module: it knows where the pipeline's files are and draws them.
import _helpers as h  # noqa: E402

products = h.available_products()
if not products:
    raise RuntimeError(
        f"No built products under {h.NETWORKS_DIR}. Run `{h.BUILD_ALL}` (or set RUN_BUILD = True above) and re-run."
    )
print("Built products:")
display(h.list_products())

# >>> pick which product to inspect (change the index) <<<
PRODUCT_INDEX = 0
PRODUCT = products[PRODUCT_INDEX]
print("\nSelected PRODUCT =", PRODUCT)

## Summary

Counts, total line length and the manifest's note on which electrical values are real and which are placeholders.

In [ ]:
summary = h.summarise(PRODUCT)
for key in ("product", "methodology", "nodes", "edges", "line_length_km", "crs", "electrical_values_note"):
    print(f"{key:24s} {summary[key]}")
print("\nnodes by kind:  ", summary["node_kinds"])
print("edges by source:", summary["edge_sources"])

## Interactive map

Pan, zoom and **hover** over buses for attributes (Plotly on an OpenStreetMap basemap, no API key). Every edge is drawn, one colour per source (tab10): for the inferred products the dense OSM road mesh *is* the distribution network, drawn under the transmission / backbone / anchor layers. Only substation and generator buses are marked. Pass `roads=False` for a power-only view, or `clip=h.MAURITIUS_BBOX` to focus on one island (the two islands are ~560 km apart, so the full extent of an inferred product is mostly ocean). `h.plot_network(PRODUCT)` gives a static matplotlib version.

In [ ]:
h.explore_network(PRODUCT)

## Node and edge tables

The GeoParquet layers are what the viewer shows. Nodes carry `bus_id`, `kind` (`substation`, `generator`, `transformer`, `distribution_node`; `junction` in the base product), `source`, `model_v_nom_kv` and, for power assets connected to a road, `anchor_status` / `anchor_distance_m` (how far the asset sat from the road it was joined to). Edges carry `source` (for example `osm` road links, `provided_transmission`, `substation_anchor` / `generator_anchor` connectors), `bus0` / `bus1`, `length_km` and `model_v_nom_kv`. `is_inferred` marks what the inference added. In the inferred products `v_nom_kv` and `s_nom_mva` are deliberately empty: the electrical values are placeholders, kept in `model_v_nom_kv` / `model_s_nom_mva`.

In [ ]:
nodes, edges = h.load_layers(PRODUCT)
node_columns = [
    "bus_id",
    "name",
    "kind",
    "source",
    "model_v_nom_kv",
    "anchor_status",
    "anchor_distance_m",
    "is_inferred",
]
edge_columns = ["line_id", "source", "bus0", "bus1", "length_km", "model_v_nom_kv", "model_s_nom_mva", "is_inferred"]
print("Power terminals (nodes that are not inferred road vertices):")
display(h.anchor_nodes(nodes)[node_columns].head(10))
print("Edges:")
display(edges[edge_columns].head(10))
print("Edges by source:")
display(edges.groupby("source", dropna=False)["length_km"].agg(count="size", length_km="sum").round(1))

## Validation report

`data/out/energy/<product>/validation.json` records advisory checks: totals, an `errors` list (empty for a usable network) and `warnings`, for example when the model's line length or generation capacity differs from CEB's published totals by more than the configured tolerance. The build logs the same warnings, so a run is never silently "fine". `status` is `valid`, `valid_with_warnings` or `invalid`.

In [ ]:
if h.found(h.OUT_DIR / PRODUCT / "validation.json", h.BUILD_ALL):
    validation = h.load_validation(PRODUCT)
    print("status:", validation["status"])
    print("totals:", validation["totals"])
    for error in validation.get("errors", []):
        print("  error:  ", error)
    for warning in validation.get("warnings", []):
        print("  warning:", warning)
    for check, result in validation.get("checks", {}).items():
        print(f"  check {check}: {result.get('status')}")

## PyPSA view

The same topology as a PyPSA network (`<product>.nc`), for electrical inspection: buses, lines, transformers (where the inferred-provided product joins the CEB 66 kV backbone to the 11 kV distribution placeholder) and generators. Capacities are fixed; nothing is extendable.

In [ ]:
n = h.load_pypsa(PRODUCT)
print(n)
print(f"{len(n.buses)} buses, {len(n.lines)} lines, {len(n.transformers)} transformers, {len(n.generators)} generators")
display(n.buses.head())
display(n.lines.head())